# Session 08: Asynchronous Python

**Notebook 1 of 3: Concepts** · used during the live session and for revision afterwards (about 70 minutes on your own)

Much of an AI application's time is spent **waiting**: for an LLM to answer, for a vector store, for another team's API. Waiting one request at a time wastes most of that time. This session shows how Python waits for many things at once with `async` and `await`, when that helps, and when it doesn't.

By the end of this notebook you will be able to:

1. **Explain** the event loop, and say when async helps (waiting on I/O) and when it doesn't (CPU-bound work).
2. **Write** coroutines with `async def` and `await`, and **run** them concurrently with `asyncio.gather` and `TaskGroup`.
3. **Make** concurrent HTTP calls with `httpx.AsyncClient`, sharing one client.
4. **Apply** timeouts with `asyncio.timeout`, and **handle** cancellation and partial failure.
5. **Choose** between `def` and `async def` endpoints in FastAPI, and **avoid** blocking the event loop.
6. **Test** asynchronous code with pytest-asyncio.

> **Kernel:** the course environment, after `uv sync`. Everything runs offline: HTTP calls go to a fake transport that waits, like a real network would.
>
> **Notebooks are different from scripts.** Jupyter already runs an event loop, so here you can write `await` directly in a cell. In a script, you start the loop yourself with `asyncio.run(main())`. Part 1 shows both.

## The engineering problem

The on-call dashboard you'll build in the workbook shows, for each team, its open tickets **and** the assets it owns, which come from the Asset Inventory API from Session 05. There are four teams, so four HTTP calls. Each takes about half a second, almost all of it waiting for the network.

Measured on the workbook's service with the fake inventory set to 0.5 seconds of latency:

| Approach | `GET /dashboard` |
|---|---:|
| One team after another | about 2.3 s |
| All four at once | about 0.8 s |

With 40 teams, the first approach takes over 20 seconds, and the second still takes under a second, as long as the inventory can serve them all at once. The work is the same; the difference is only in how the **waiting** is arranged.

In Java you'd reach for a thread pool or `CompletableFuture`, in C# for `async`/`await` with `Task.WhenAll`, in JavaScript for `Promise.all`. Python's version is `asyncio`, and its syntax will look familiar. Its behaviour has some important differences, which this notebook points out.

## Part 1: Coroutines

`async def` defines a **coroutine function**. Calling it does **not** run its body: it returns a **coroutine object**, a paused computation. `await` runs it and gives you its result.

In [ ]:
import asyncio
import time


async def fetch_assets(team: str) -> list[str]:
    print(f"  fetching {team}...")
    await asyncio.sleep(0.3)  # stands in for waiting on the network
    return [f"{team}-asset-1", f"{team}-asset-2"]


pending = fetch_assets("network")  # nothing printed: the body hasn't started
print("called:", pending)

assets = await pending  # now it runs
print("awaited:", assets)

**Explain:** the first `print` shows a `coroutine object`, and "fetching network..." appears only after `await`. A coroutine is a recipe for work, not the work itself.

This is a real difference from other languages:

| | JavaScript | C# | Python |
|---|---|---|---|
| Calling an async function | Starts it immediately; returns a running `Promise` | Starts it immediately; returns a running `Task` | **Doesn't start it**; returns a coroutine object |
| Run several at once | `Promise.all([...])` | `Task.WhenAll(...)` | `asyncio.gather(...)` or a `TaskGroup` |
| Forgetting `await` | The work still happens, unobserved | The work still happens, unobserved | **The work never happens**, and Python warns "coroutine ... was never awaited" |
| Entry point | The runtime's event loop is always there | `async Task Main()` | `asyncio.run(main())` |

**In a script**, the same code needs an entry point, because nothing is running an event loop yet:

```python
import asyncio

async def main() -> None:
    assets = await fetch_assets("network")
    print(assets)

if __name__ == "__main__":
    asyncio.run(main())    # creates the event loop, runs main() to completion, closes the loop
```

In this notebook, Jupyter's loop is already running, so `asyncio.run()` would fail with "cannot be called from a running event loop". The debugging lab (notebook 3) reproduces that error.

## Part 2: The event loop

**Predict:** how long do these two cells take?

In [ ]:
started = time.perf_counter()
for team in ["database", "network", "platform"]:
    await fetch_assets(team)
print(f"one after another: {time.perf_counter() - started:.2f}s")

In [ ]:
started = time.perf_counter()
results = await asyncio.gather(
    *(fetch_assets(team) for team in ["database", "network", "platform"])
)
print(f"gathered: {time.perf_counter() - started:.2f}s")
print(results)

**Explain:** the first loop awaits each coroutine before creating the next, so the waits add up: about 0.9 s. `asyncio.gather` starts all three, and while each is waiting in `asyncio.sleep`, the others run, so the waits **overlap**: about 0.3 s. The results come back in the order of the arguments, not the order of completion.

![The event loop](../../../assets/diagrams/async-event-loop.svg)

**How this works**

- The **event loop** runs on **one thread** and runs **one coroutine at a time**. There's no parallelism here: just very fast switching between coroutines.
- A coroutine runs until it reaches an `await` on something that isn't ready yet (a network response, a timer). Then it's **parked**, and the loop runs the next ready coroutine. Switching happens **only at `await`**, so between two `await`s your code is never interrupted by other coroutines.
- The **operating system** does the actual waiting, for all sockets and timers at once, and tells the loop which are ready. The loop then resumes the coroutines waiting on them.
- A parked coroutine costs almost nothing, which is why one thread can keep thousands of requests in flight, far more than you'd want threads for.
- The catch: a call that blocks without `await` (`time.sleep`, `httpx.get`, `requests.get`, a heavy calculation) never gives control back. **The whole loop stops**, along with every other coroutine and every other request to your service.

### Tasks: starting work now, collecting it later

`asyncio.create_task()` schedules a coroutine to run **as soon as the loop is free**, and returns a `Task` you can await later. It's the closest thing to JavaScript's eager promises. `gather` creates tasks for you.

In [ ]:
async def log(message: str, delay: float) -> str:
    await asyncio.sleep(delay)
    print(f"{time.perf_counter() - started:.2f}s  {message}")
    return message


started = time.perf_counter()
slow = asyncio.create_task(log("slow task finished", 0.3))  # starts running at the next await
fast = asyncio.create_task(log("fast task finished", 0.1))
print(f"{time.perf_counter() - started:.2f}s  tasks created, doing other work")
await asyncio.sleep(0.2)
print(f"{time.perf_counter() - started:.2f}s  main code awaiting the slow task")
print("result:", await slow, "/", await fast)

**Engineering Insight:** keep a reference to every task you create, and await it. A task that nobody awaits can fail silently, and the event loop only keeps weak references to tasks, so an unreferenced task can even be garbage-collected before it finishes. `gather` and `TaskGroup` (Part 5) take care of this for you; prefer them over bare `create_task`.

## Part 3: Concurrent HTTP with `httpx.AsyncClient`

`httpx.AsyncClient` is the asynchronous twin of the `httpx.Client` from Session 05: the same methods, but each one is a coroutine. Here it talks to a fake inventory through `MockTransport`. The handler is a coroutine that waits 0.3 s, so the timings behave like a real network:

In [ ]:
import httpx

ASSETS = {
    "database": ["db-03", "db-backup"],
    "it-support": ["badge-zrh"],
    "network": ["vpn-ber", "dns-int"],
    "platform": ["api-gw-01", "track-web", "ci-runner-02"],
}
in_flight = 0
peak_in_flight = 0


async def fake_inventory(request: httpx.Request) -> httpx.Response:
    global in_flight, peak_in_flight
    in_flight += 1
    peak_in_flight = max(peak_in_flight, in_flight)
    await asyncio.sleep(0.3)  # network latency
    in_flight -= 1
    team = request.url.params["team"]
    return httpx.Response(200, json={"team": team, "items": ASSETS.get(team, [])})


inventory = httpx.AsyncClient(
    base_url="http://inventory.test", transport=httpx.MockTransport(fake_inventory), timeout=5.0
)


async def team_assets(client: httpx.AsyncClient, team: str) -> list[str]:
    response = await client.get("/assets", params={"team": team})
    response.raise_for_status()
    return response.json()["items"]


teams = list(ASSETS)

started = time.perf_counter()
sequential = [await team_assets(inventory, team) for team in teams]
print(f"sequential: {time.perf_counter() - started:.2f}s, peak in flight {peak_in_flight}")

peak_in_flight = 0
started = time.perf_counter()
concurrent = await asyncio.gather(*(team_assets(inventory, team) for team in teams))
print(f"concurrent: {time.perf_counter() - started:.2f}s, peak in flight {peak_in_flight}")
print(dict(zip(teams, concurrent)))

![Sequential versus concurrent I/O](../../../assets/diagrams/sequential-vs-concurrent-io.svg)

**How this works**

- Each request is a short burst of Python work (build the request, later parse the JSON) around a long wait. The blue bursts are tiny; the waiting is almost everything.
- **Sequentially**, each request starts only when the previous one has finished, so the total is the **sum** of the requests: 4 × 0.3 s here.
- With **`gather`**, all four are sent immediately, and the waits overlap. The total is roughly the **slowest** request. The "peak in flight" counter shows all four were waiting at once.
- The blue bursts still never overlap: one thread runs Python one piece at a time. That's why async speeds up **waiting**, and does nothing for **computing** (Part 7).
- All four requests share **one** `AsyncClient`, and so its connection pool. Creating a client per request throws that pool away, and is one of the mistakes in the debugging lab.

**Modify:** add 36 more teams (`teams = [f"team-{n}" for n in range(40)]`) and run both versions. Predict the timings first.

### Limiting concurrency

"All at once" is fine for four requests. For 4,000, you'd overwhelm the other service, or hit its rate limit. An `asyncio.Semaphore` caps how many coroutines can be inside a block at the same time:

In [ ]:
limit = asyncio.Semaphore(2)  # at most two requests in flight


async def polite_team_assets(client: httpx.AsyncClient, team: str) -> list[str]:
    async with limit:
        return await team_assets(client, team)


peak_in_flight = 0
started = time.perf_counter()
await asyncio.gather(*(polite_team_assets(inventory, team) for team in teams))
print(f"limited to 2: {time.perf_counter() - started:.2f}s, peak in flight {peak_in_flight}")

Four requests, two at a time: two rounds of 0.3 s. `httpx.AsyncClient` also limits connections per client (`httpx.Limits(max_connections=...)`), but a semaphore states the intent in your code and works for any kind of awaitable.

## Part 4: Timeouts and cancellation

An external call without a deadline can hang your request forever. Session 05 set **HTTPX timeouts**, which limit each phase of **one** HTTP request (connecting, reading ...). `asyncio.timeout` limits a **whole block of async code**, however many requests it makes, for example all the pages of a paginated listing:

In [ ]:
async def slow_inventory(request: httpx.Request) -> httpx.Response:
    await asyncio.sleep(2.0)
    return httpx.Response(200, json={"items": []})


slow_client = httpx.AsyncClient(
    base_url="http://inventory.test", transport=httpx.MockTransport(slow_inventory)
)

started = time.perf_counter()
try:
    async with asyncio.timeout(0.5):
        await team_assets(slow_client, "network")
except TimeoutError:
    print(f"gave up after {time.perf_counter() - started:.2f}s")

**Explain:** when the deadline passes, `asyncio.timeout` **cancels** the code inside the block: the `await` that was waiting raises `asyncio.CancelledError` inside the coroutine, and the `async with` block turns that into a `TimeoutError` for you. The request was abandoned at 0.5 s, not 2 s.

Three facts about cancellation that matter in practice:

1. **It only happens at an `await`.** Code that never awaits can't be cancelled; the debugging lab shows a timeout that doesn't work for that reason.
2. **`CancelledError` is a `BaseException`, not an `Exception`.** So `except Exception:` doesn't catch it, and cancellation passes through your error handling, as it should. If you ever catch it to clean up, re-raise it.
3. **Clean up with `try`/`finally` or `async with`**, which run during cancellation too. That's how `AsyncClient` closes its connections even when a request is cancelled.

| | Limits | Raises |
|---|---|---|
| HTTPX `timeout=5.0` | Each phase of one HTTP request | `httpx.TimeoutException` (a subclass of `httpx.TransportError`) |
| `asyncio.timeout(10)` | Everything inside the block | `TimeoutError` |

Use both: HTTPX timeouts so no single request hangs, and an overall deadline for the operation your caller is waiting for. The workbook's `inventory_timeout_seconds` and `inventory_deadline_seconds` settings are exactly this pair.

## Part 5: When some calls fail

Real services fail. What should happen to the dashboard when the inventory fails for **one** team? Python offers three behaviours, and you should choose deliberately.

In [ ]:
async def flaky_inventory(request: httpx.Request) -> httpx.Response:
    if request.url.params["team"] == "network":  # fails fast
        return httpx.Response(503, json={"error": "maintenance"})
    await asyncio.sleep(0.3)  # the others take a while
    return httpx.Response(200, json={"items": ASSETS[request.url.params["team"]]})


flaky = httpx.AsyncClient(
    base_url="http://inventory.test", transport=httpx.MockTransport(flaky_inventory)
)

# 1. Plain gather: the first exception propagates; the other results are lost to you
try:
    await asyncio.gather(*(team_assets(flaky, team) for team in teams))
except httpx.HTTPStatusError as error:
    print("1. gather:", error.response.status_code, "- and no results at all")

# 2. return_exceptions=True: exceptions come back in the list, next to the results
results = await asyncio.gather(
    *(team_assets(flaky, team) for team in teams), return_exceptions=True
)
for team, result in zip(teams, results):
    print(
        "2.",
        f"{team:<11}",
        "FAILED " + type(result).__name__ if isinstance(result, Exception) else result,
    )

In case 1, the other three requests **kept running** after `gather` raised; nobody was waiting for their results any more. With `return_exceptions=True` you **must** check each result's type; it's easy to forget, and then an exception object flows on as if it were data.

### `TaskGroup`: all or nothing

Python 3.11 added `asyncio.TaskGroup`, the structured way to run tasks together. If any task fails, the group **cancels the others** and raises all the errors together as an `ExceptionGroup`, which you handle with `except*`:

In [ ]:
try:
    async with asyncio.TaskGroup() as group:
        tasks = {team: group.create_task(team_assets(flaky, team)) for team in teams}
except* httpx.HTTPStatusError as errors:
    print("3. TaskGroup failed:", [e.response.status_code for e in errors.exceptions])
    print(
        "   other tasks:",
        {team: ("cancelled" if t.cancelled() else "finished") for team, t in tasks.items()},
    )

### Which one?

| Behaviour | Use when |
|---|---|
| `TaskGroup` (or plain `gather`): fail as a whole | The results only make sense together; a partial answer would be wrong. Prefer `TaskGroup`: it also cancels the rest |
| Handle failure **per item**, inside each coroutine | A partial answer is useful: show what you have, and say what's missing |
| `gather(..., return_exceptions=True)` | The same, when you can't change the coroutines; check every result's type |

The dashboard wants a partial answer: if the inventory fails for one team, the on-call engineer should still see the other three. The workbook wraps each team's fetch in a function that **turns expected failures into data** (`TeamAssets(error=...)`) and lets **unexpected** exceptions (bugs) propagate:

In [ ]:
async def fetch_one(client: httpx.AsyncClient, team: str, *, deadline: float) -> dict:
    try:
        async with asyncio.timeout(deadline):
            return {"team": team, "assets": await team_assets(client, team)}
    except TimeoutError:
        return {"team": team, "error": f"no answer within {deadline:g}s"}
    except httpx.HTTPError as error:  # expected: the inventory's problems
        return {"team": team, "error": str(error).splitlines()[0]}


for row in await asyncio.gather(*(fetch_one(flaky, team, deadline=1.0) for team in teams)):
    print(row)

**Engineering Insight:** notice what this pattern does **not** catch: a `KeyError` from a bug in `team_assets` would still propagate and fail the request with a `500`, which is what you want. Catch the failures you **expect** from the outside world; let the rest be loud.

## Part 6: `async` in FastAPI

FastAPI runs on an event loop, and accepts both kinds of endpoint:

| Endpoint | Where it runs | Rule |
|---|---|---|
| `def endpoint()` | In a **thread pool**, so blocking calls are fine | Use for code that blocks: sync libraries, file I/O, CPU work |
| `async def endpoint()` | **On the event loop** | Use when you `await` async libraries. **Never block** inside it |

**Predict:** each endpoint below takes 0.3 s. How long do four simultaneous requests take, for each one?

In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/async-await")
async def async_await() -> dict:
    await asyncio.sleep(0.3)  # yields to the loop while waiting
    return {"ok": True}


@app.get("/async-block")
async def async_block() -> dict:
    time.sleep(0.3)  # blocks the loop: a mistake
    return {"ok": True}


@app.get("/sync-block")
def sync_block() -> dict:
    time.sleep(0.3)  # blocks a thread-pool thread, not the loop
    return {"ok": True}


async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://api") as api:
    for path in ["/async-await", "/async-block", "/sync-block"]:
        started = time.perf_counter()
        await asyncio.gather(*(api.get(path) for _ in range(4)))
        print(f"{path:<13} 4 concurrent requests: {time.perf_counter() - started:.2f}s")

**Explain:**

- `/async-await`: about 0.3 s. All four wait concurrently on the loop.
- `/async-block`: about 1.2 s. `time.sleep` in an `async def` **holds the loop**, so the four requests run one after another, and so would every other request to the service, including `/health`. This is the most damaging async mistake, because it looks fine in a single-request test.
- `/sync-block`: about 0.3 s. FastAPI ran each in a pool thread, so the loop stayed free.

(`httpx.ASGITransport` sends requests straight into the app on the current event loop, like an asynchronous `TestClient`.)

**Rule of thumb:** if an endpoint calls anything that blocks and has no async version, make it a plain `def`. If it awaits async clients (the workbook's inventory client, an async LLM SDK in Session 09), make it `async def`, and make sure everything inside it either awaits or is quick.

If you must call a blocking function from async code, run it in a thread: `result = await asyncio.to_thread(blocking_function, arg)`.

### One client for the whole app

An `AsyncClient` owns a connection pool. Create **one** at startup and close it at shutdown, which is what the lifespan from Session 07 is for:

```python
@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    app.state.inventory = InventoryClient.from_settings(settings)   # one shared client
    try:
        yield
    finally:
        if app.state.inventory is not None:
            await app.state.inventory.aclose()                       # release its connections
```

Endpoints receive it through a dependency, as the `TicketService` did in Session 07, so tests can override it with a fake.

## Part 7: CPU-bound work and the GIL

Async overlaps **waiting**. It does nothing for **computing**:

In [ ]:
import hashlib


def checksum(data: bytes, rounds: int = 200_000) -> str:
    digest = data
    for _ in range(rounds):
        digest = hashlib.sha256(digest).digest()
    return digest.hex()[:12]


async def checksum_async(data: bytes) -> str:
    return checksum(data)  # no await inside: nothing to overlap


started = time.perf_counter()
checksum(b"a")
one = time.perf_counter() - started

started = time.perf_counter()
await asyncio.gather(*(checksum_async(bytes([n])) for n in range(4)))
print(
    f"one checksum: {one:.2f}s; four 'concurrent' checksums: {time.perf_counter() - started:.2f}s"
)

**Explain:** four times as long, because there was never any waiting to overlap. Making a function `async def` doesn't make it concurrent; only `await`ing something that waits does.

Your options for CPU-bound work in Python:

| Tool | What it does | Notes |
|---|---|---|
| `asyncio.to_thread(f, ...)` / threads | Runs `f` in another thread | Keeps the event loop responsive. In standard CPython, the **GIL** (global interpreter lock) lets only one thread run Python bytecode at a time, so pure-Python CPU work doesn't get faster. C libraries that release the GIL (NumPy, hashlib on large inputs) can |
| `concurrent.futures.ProcessPoolExecutor` | Runs `f` in other processes | True parallelism for pure-Python CPU work; arguments and results are copied between processes |
| A specialised library | NumPy, vectorised operations | Usually the biggest win for numeric work |

Python 3.13 also ships an optional, experimental **free-threaded** build without the GIL. The course uses the standard build; for this course, the GIL means: **async and threads for waiting, processes or native libraries for heavy computing.** In AI applications, the heavy computing (model inference) usually happens in another service or a native library, and your Python code mostly waits, which is why async fits so well.

## Part 8: Testing async code

pytest can't await a coroutine by itself. The **pytest-asyncio** plugin runs `async def` tests on an event loop when you mark them with `@pytest.mark.asyncio` (or mark a whole file with `pytestmark = pytest.mark.asyncio`). Here's a small project tested with real pytest:

In [ ]:
import os
import pathlib
import subprocess
import sys
import tempfile
import textwrap

MODULE = """
import asyncio

import httpx


async def team_assets(client: httpx.AsyncClient, team: str) -> list[str]:
    response = await client.get("/assets", params={"team": team})
    response.raise_for_status()
    return response.json()["items"]


async def all_assets(client: httpx.AsyncClient, teams: list[str]) -> dict[str, list[str]]:
    results = await asyncio.gather(*(team_assets(client, team) for team in teams))
    return dict(zip(teams, results))
"""

TESTS = """
import asyncio

import httpx
import pytest

from inventory import all_assets, team_assets

pytestmark = pytest.mark.asyncio


class Fake:
    def __init__(self):
        self.in_flight = 0
        self.peak = 0

    async def handler(self, request):
        self.in_flight += 1
        self.peak = max(self.peak, self.in_flight)
        await asyncio.sleep(0.05)
        self.in_flight -= 1
        if request.url.params["team"] == "broken":
            return httpx.Response(500)
        return httpx.Response(200, json={"items": [request.url.params["team"] + "-1"]})


@pytest.fixture
def fake():
    return Fake()


async def test_one_team(fake):
    async with httpx.AsyncClient(base_url="http://t", transport=httpx.MockTransport(fake.handler)) as c:
        assert await team_assets(c, "network") == ["network-1"]


async def test_teams_are_fetched_concurrently(fake):
    async with httpx.AsyncClient(base_url="http://t", transport=httpx.MockTransport(fake.handler)) as c:
        result = await all_assets(c, ["a", "b", "c"])
    assert result == {"a": ["a-1"], "b": ["b-1"], "c": ["c-1"]}
    assert fake.peak == 3


async def test_server_error_is_raised(fake):
    async with httpx.AsyncClient(base_url="http://t", transport=httpx.MockTransport(fake.handler)) as c:
        with pytest.raises(httpx.HTTPStatusError):
            await team_assets(c, "broken")
"""

env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
env.update(NO_COLOR="1", PY_COLORS="0")
with tempfile.TemporaryDirectory() as workdir:
    folder = pathlib.Path(workdir)
    (folder / "inventory.py").write_text(textwrap.dedent(MODULE), encoding="utf-8")
    (folder / "test_inventory.py").write_text(textwrap.dedent(TESTS), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
        cwd=folder,
        capture_output=True,
        text=True,
        env=env,
    )
print(result.stdout)

**Explain:**

- The fake counts requests in flight, so `test_teams_are_fetched_concurrently` proves concurrency **without measuring time**. Timing assertions are slow and can be flaky on a busy machine; a counter is deterministic. The workbook's `FakeInventory` uses the same idea.
- `MockTransport` accepts an `async def` handler, so the fake can `await asyncio.sleep`, just as a real network makes the client wait.
- A test that forgets `@pytest.mark.asyncio` (with pytest-asyncio's default *strict* mode) fails with "async def functions are not natively supported", so a missing mark can't pass by accident.
- FastAPI endpoints, `async def` or not, are still tested with the synchronous `TestClient` from Session 07. It runs the app's event loop for you.

Finally, close the clients this notebook created:

In [ ]:
for client in (inventory, slow_client, flaky):
    await client.aclose()
print("closed")

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| Calling a coroutine without `await` | Nothing runs; a coroutine object flows on, plus a "never awaited" warning | `await` it, or pass it to `gather` / a `TaskGroup` |
| `time.sleep`, `requests`, sync `httpx` inside `async def` | The whole event loop stops; every request waits | `await asyncio.sleep`, `httpx.AsyncClient`, or `await asyncio.to_thread(...)`; or make the endpoint a plain `def` |
| `asyncio.run()` inside a running loop (Jupyter, an async endpoint) | `RuntimeError` | `await` directly; `asyncio.run` only at a script's entry point |
| Awaiting in a loop when the calls are independent | The waits add up | `asyncio.gather` or a `TaskGroup` |
| A new `AsyncClient` per request | Connection pools thrown away; sockets left open if not closed | One shared client, created at startup and closed at shutdown |
| No deadline on external calls | One slow dependency hangs your endpoint | HTTPX timeouts, plus `asyncio.timeout` around the whole operation |
| `return_exceptions=True` without checking results | Exception objects are treated as data | Check each result's type, or handle failures inside each coroutine |
| `except Exception` around everything | Bugs are hidden as "partial failures" | Catch the external failures you expect; let bugs propagate |
| Expecting `async` to speed up CPU work | No speed-up at all | Processes, native libraries, or another service |
| Shared state updated across an `await` | Lost updates when coroutines interleave | Don't hold read-modify-write across an `await`; use `asyncio.Lock` if you must |

## Practical engineering application

AI services are dominated by waiting: an LLM call takes seconds, retrieval and other APIs add more. From Session 09 on, you'll call LLM SDKs that offer async clients; the pattern from today carries straight over. **One shared async client, created at startup; concurrent calls with `gather` or a `TaskGroup`; a deadline on everything external; a deliberate choice between all-or-nothing and partial results; and no blocking calls inside `async def`.**

## Key takeaways

- `async def` creates a coroutine; nothing runs until it's awaited. Forgetting `await` means the work never happens.
- One event loop, one thread, one coroutine at a time; switching only at `await`. Async overlaps **waiting**, not computing.
- `asyncio.gather` and `TaskGroup` run coroutines concurrently; a `Semaphore` limits how many at once.
- `asyncio.timeout` puts a deadline on a block; cancellation happens at `await`s, and `CancelledError` isn't an `Exception`.
- Choose failure behaviour deliberately: all or nothing (`TaskGroup`), or per-item results with expected failures as data.
- In FastAPI, `def` endpoints run in threads, and `async def` endpoints run on the loop and must never block. Share one async client through the lifespan.
- Test async code with pytest-asyncio, and prove concurrency with counters rather than timings.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** the **`03-async-debugging-lab.ipynb`** notebook has six broken async programs to diagnose. Then test yourself with **`02-practice-quiz.ipynb`**, and finish **Project 02** with the workbook.